# Solutions — Routing

Only look here after you've actually tried the exercises in `router.ipynb`.

Most of this topic happens in `react-scratch`, so most answers are explanations rather than
cells. LESSON 61's helpers are plain JavaScript and do run here.

### LESSON 58 — Exercise

**1-2. Setting up.** Typing `/about` directly works because `<BrowserRouter>` reads the real
URL; the router is not inventing a URL scheme of its own. Clicking a `<Link>` never spins the
reload indicator, because no document is requested.

**3. The counter — the whole point of the topic.**

With `<Link>`: the counter keeps its value. No document was loaded, the component never
unmounted, and state survived.

With `<a href>`: the counter **resets to zero**. The browser threw the page away and started
again — new JavaScript, new React, new everything. That is not React losing state; it is the
page ceasing to exist.

Nothing else in this course demonstrates the difference between "the app changed what it shows"
and "the app was destroyed and rebuilt" as directly.

**4. `NavLink` and `isActive`.** Without it you would have to: keep a `currentPage` in state,
set it in every link's handler, keep it in step with the URL when the user uses Back or Forward
or types an address, and reset it correctly on first load from whatever URL they arrived at.

That is four places to get wrong, replaced by a function the router already has the answer for.
It is LESSON 29 again — a value you can derive should not be stored.

**5. The unlinked route and the nonsense URL.** `/contact` renders fine; a route does not need a
link, it needs a match. `/nonsense` renders **nothing** — a blank page, no error, no message.
What is missing is a catch-all route, which is LESSON 60.

**Common mistakes.**

- Using `<a href>` inside the app because it "works". It works by destroying the app.
- Putting `<BrowserRouter>` inside a component that re-renders, rather than once at the root.
- Expecting `<Routes>` to render every match. It picks one.
- Installing `react-router-dom` after copying a tutorial. It stopped at v7; v8 is `react-router`.

### LESSON 58 — Mini challenge

**1. "Routing is just `useState` holding the page name."** Two things it would not do: the URL
would never change, so nothing could be **bookmarked or shared**; and the **Back button** would
either do nothing or leave the site entirely. One thing it would break: opening
`yoursite.com/about` directly would show the home page, because the state starts at its initial
value regardless of the address.

**2. Why `<Link>` renders a real `<a>`.** Because an anchor is not just a click target. It
gives middle-click "open in new tab", right-click "copy link address", Ctrl/Cmd-click,
keyboard focus and activation with Enter, and a screen-reader announcement that this is a link
to somewhere. A `<span onClick>` has none of that, and re-implementing even half of it correctly
is more work than using the right element.

**3. When `<a href>` is correct.** Leaving your app: an external site, a `mailto:`, a file
download, or a link to a different application on another domain. The rule is that `<a>` is for
leaving and `<Link>` is for moving within.

**4. The hand-rolled active state.** Two bugs `NavLink` cannot have: it can go **stale** — the
user navigates with the Back button and `currentPage` still says the old page, so the wrong item
is highlighted; and it can be **wrong on first load** — arriving directly at `/about` leaves
`currentPage` at its initial value, so nothing is highlighted or the wrong thing is. Both are
the duplicated-state problem from LESSON 29.

### LESSON 59 — Exercise

**1. Dynamic segments.** `/teams/17` gives `params.teamId === "17"` — a **string**. So does
`/teams/anything`: the segment matches any characters, and validating the content is your job,
not the router's.

**2. Converting and handling a miss.** `Number(params.teamId)`, then a `find`. `/teams/999`
matches the route perfectly and finds no team, so the component must handle it — an
"unknown team" message, not a crash. This is LESSON 46's empty state, reached through a URL.

**3. Nesting.** With the layout as a parent and the detail as a child, moving between
`/teams/1` and `/teams/2` re-renders only the outlet. A counter in the layout keeps its value,
because that component never unmounted — only the thing inside `<Outlet />` changed.

**4. The index route.** Without it, `/teams` renders the layout and an **empty outlet**: the
heading and the link list appear with a blank space where the detail belongs. With it, "Pick a
team" fills that space. It is the commonest nested-routing bug precisely because nothing errors.

**5. The selection in the URL.** What would improve: a colleague could be sent a link to a
specific person; the Back button would step through the people you looked at; and a refresh
would keep the selection instead of resetting the screen.

What would get harder: the component now has to cope with a URL naming somebody who does not
exist — a case that `useState` made impossible, because the only way to select someone was to
click a real row.

### LESSON 59 — Mini challenge

**A — the param name does not match.** The route declares `:id`; the component destructures
`userId`. `useParams()` returns `{ id: "..." }`, so `userId` is `undefined` and the page renders
an empty `<p>`. No error — the name in the path and the name you read must be the same word.

**B — no `<Outlet />`.** The nesting is correct and `/settings/profile` does match, so React
Router renders `Settings`… which has nowhere to put the child. The user sees only the heading.
The child is matched and then dropped on the floor.

**C — two separate failures.**

*Types:* `useParams` gives a string, and if `products` have numeric ids then
`p.id === id` compares `1 === "1"`, which is `false`. The find returns `undefined` and
`product.name` throws. It "works in development" only if the test data happened to use string
ids.

*Missing data:* even with the types fixed, a URL can name a product that does not exist —
someone's old bookmark, a deleted item, a typo. `find` returns `undefined` and the component
crashes on `.name`. The route matched; the data did not. A guard before the render is the only
answer, and it is the same guard as `/teams/999`.

### LESSON 60 — Exercise

**1. The catch-all.** Before: `/nonsense` renders a blank page — no error, nothing to click, and
nothing in the console. After: the `NotFound` component with a way back. The blank page is worse
than an error because it is indistinguishable from a crash.

**2-3. Where `navigate` goes.** After the guard, the form behaves: an empty submit shows the
messages and keeps what was typed; a valid submit moves on.

Before the guard, the user is thrown to another page **whether or not the submit worked** — and
their input is gone, because the form component unmounted. They do not see the error message,
because the component that would have shown it no longer exists. It is worse than a missing
error message: a missing message leaves them looking at their data wondering why nothing
happened, whereas this loses the data and gives them no reason.

**4. The button versus the link.** The button cannot: be middle-clicked to open in a new tab, be
right-clicked for "copy link address", or be Ctrl/Cmd-clicked. It is also announced as a button
rather than a link to assistive technology, and it has no `href` for anything to inspect. All of
that is what "they provide a better default user experience" means in React Router's warning.

**5. `/teams/999`.** The check goes **inside the component**, after the lookup. The router
cannot do it because the route *did* match — `999` is a perfectly valid segment. Only your code
knows which ids exist, and it may not know until after a fetch.

### LESSON 60 — Mini challenge

| | | why |
|---|---|---|
| 1. quiz advances on a timer | **`useNavigate`** | nobody clicked |
| 2. quiz advances on a click | **`<Link>`** | the user clicked a thing that goes somewhere |
| 3. back to the list after creating | **`useNavigate`** | the submit completed; there was no navigation click |
| 4. breadcrumbs | **`<Link>`** | they are links |
| 5. session expired → `/login` | **`useNavigate`** | caused by time, not interaction |
| 6. table row opens a detail page | **`<Link>`** | see below |
| 7. cancel a form and go back | **`<Link>`** | it is a click that goes somewhere |

**1 versus 2 in one sentence:** if the user clicked the thing that takes them elsewhere, it is a
link; if something else decided, it is `useNavigate`.

**Number 6, and the three things it breaks.** A `<tr onClick={() => navigate(...)}>` loses:
**opening in a new tab** (middle-click, Ctrl/Cmd-click — people open several rows at once all
the time); **keyboard access**, because a `<tr>` is not focusable and cannot be activated with
Enter; and **the status bar and right-click menu**, so the user cannot see or copy where the row
goes.

The fix is not more JavaScript — it is putting a real `<Link>` inside the row, around the cell
content.

### LESSON 61 — Exercise

**Part 1.**

In [ ]:
const L61_DEFAULTS = { q: "", department: "all", page: 1, sort: "name" };

function l61readFilters(search) {
  const params = new URLSearchParams(search);
  return {
    q: params.get("q") ?? L61_DEFAULTS.q,
    department: params.get("department") ?? L61_DEFAULTS.department,
    page: Number(params.get("page") ?? L61_DEFAULTS.page),
    sort: params.get("sort") ?? L61_DEFAULTS.sort,
  };
}

function l61toSearch(filters) {
  const params = new URLSearchParams();
  for (const key of Object.keys(L61_DEFAULTS)) {
    const value = filters[key];
    // omit anything that equals its default - compare as strings, since the URL is strings
    if (String(value) !== String(L61_DEFAULTS[key])) {
      params.set(key, String(value));
    }
  }
  return params.toString();
}

console.log("empty string ->", JSON.stringify(l61readFilters("")));
console.log("full string  ->", JSON.stringify(l61readFilters("q=ada&department=research&page=3&sort=role")));
console.log("page is a number?", typeof l61readFilters("page=3").page);

console.log("");
console.log("all defaults ->", JSON.stringify(l61toSearch(L61_DEFAULTS)), "<- empty, as it should be");
console.log("some changed ->", l61toSearch({ q: "ada", department: "all", page: 2, sort: "name" }));

// 3 - round trip
const l61original = { q: "grace hopper", department: "research", page: 4, sort: "role" };
const l61roundTripped = l61readFilters(l61toSearch(l61original));
console.log("");
console.log("round trip   ->", JSON.stringify(l61roundTripped));
console.log("identical?   ->", JSON.stringify(l61original) === JSON.stringify(l61roundTripped));

**4. Why omit defaults.** Because the address bar is something the user reads and shares. A
directory's home page should be `/employees`, not
`/employees?q=&department=all&page=1&sort=name` — which looks broken, is impossible to read,
and is embarrassing to paste into a chat.

There is a practical benefit too: two URLs that show the same screen should *be* the same URL.
Omitting defaults means "the unfiltered list" has exactly one address rather than sixteen
equivalent ones.

**Part 2 — in `react-scratch`.**

**2. The Back button.** Changing the filter and pressing Back returns to the previous filter,
with the list updating to match. You wrote nothing for it: the filter lives in the URL, Back
restores the previous URL, and the component reads its value from there. Any state you keep in
the URL gets history for free.

**3. Search on every keystroke.** Typing "research" creates **eight** history entries, so Back
has to be pressed eight times to escape one word — and the address bar flickers with every
letter. The fix is LESSON 42's: debounce the write to the URL, keeping the input's own value in
`useState` so typing stays instant. (Writing on submit instead is also a legitimate answer, and
simpler.)

**Common mistakes.**

- Keeping a `useState` copy of a URL value "to make it easier". Now there are two sources of
  truth and the Back button desynchronises them.
- Mutating `searchParams` directly. The object changes, the URL does not, the screen does not.
- Forgetting `Number()` on a page number and getting `"2" + 1 === "21"`.
- Writing to the URL on every keystroke and destroying the Back button.
- Building a query string by hand and discovering that a search for `R&D` truncates.

### LESSON 61 — Mini challenge

| | | why |
|---|---|---|
| 1. text as the user types | **state** | it changes per keystroke; nobody links to a half-typed word |
| 2. the search that was run | **URL** | it describes what is on screen |
| 3. which page of results | **URL** | shareable, and Back should step through pages |
| 4. advanced-filters panel expanded | **state** | a viewing preference, not what is being viewed |
| 5. table sort order | **URL** | it changes what the screen shows and is worth sharing |
| 6. selected row | **either** | see below |
| 7. scroll position | **neither** | not URL, and LESSON 51 showed it is barely component state |

**1 versus 2 in one sentence:** what the user is *typing* is a draft that belongs to the input;
what the app has actually *searched for* is a description of the screen, and that is what a link
should reproduce.

**Number 6, both ways.** *For the URL:* a selected employee is a thing a colleague would want a
link to, the Back button then steps through the people you viewed, and a refresh keeps your
place — the argument from LESSON 59.

*For state:* if the selection is a lightweight preview that the user flicks through quickly, one
history entry per row makes Back useless, and nobody wants to share "the third row highlighted".

**What decides it:** whether the selection is a *destination* or a *glance*. If the detail is
substantial enough that someone would send it to a colleague, put it in the URL — and at that
point it is usually a nested route (LESSON 59) rather than a search param at all.